In [14]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
# ═══════════════════════════════════════════════════════════════════════════════
# SETUP & IMPORTS
# ═══════════════════════════════════════════════════════════════════════════════
import os
import json
import cv2
import numpy as np
import pandas as pd
import h5py
from pathlib import Path
from tqdm.auto import tqdm
from scipy.signal import savgol_filter
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

# Configuration
DATASET_DIR = r"Dataset"
PROCESSED_DIR = os.path.join(DATASET_DIR, "processed")
HDF5_DIR = os.path.join(PROCESSED_DIR, "hdf5")
METADATA_DIR = os.path.join(PROCESSED_DIR, "metadata")

# Create directories
for dir_path in [PROCESSED_DIR, HDF5_DIR, METADATA_DIR]:
    os.makedirs(dir_path, exist_ok=True)

# Processing parameters
TARGET_FPS = 30  # Consistent frame rate
FRAME_SIZE = (224, 224)  # Standard resolution for model compatibility
MAX_VELOCITY = 50.0  # pixels/frame (for outlier detection)
DT = 1.0 / TARGET_FPS  # Time delta between frames

print("✅ Setup complete")
print(f"📁 Dataset directory: {DATASET_DIR}")
print(f"📁 Processed directory: {PROCESSED_DIR}")
print(f"⚙️  Target FPS: {TARGET_FPS} | Frame size: {FRAME_SIZE}")

✅ Setup complete
📁 Dataset directory: Dataset
📁 Processed directory: Dataset\processed
⚙️  Target FPS: 30 | Frame size: (224, 224)


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TASK 1: Data Import & Format Standardization
# ═══════════════════════════════════════════════════════════════════════════════

def load_frames_from_directory(clip_dir, frame_size=FRAME_SIZE):
    """
    Load frames from a directory containing JPG images.
    
    Args:
        clip_dir: Path to clip directory containing .jpg frames
        frame_size: Target frame size (H, W)
    
    Returns:
        frames: Array of normalized frames (N, H, W, C) in [0, 1]
        frame_filenames: List of frame filenames in order
    """
    frame_files = sorted(Path(clip_dir).glob("*.jpg"))
    frames = []
    frame_filenames = []
    
    for frame_file in frame_files:
        try:
            # Read image
            img = cv2.imread(str(frame_file))
            if img is None:
                continue
            
            # Convert BGR to RGB
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            # Resize to target size
            img_resized = cv2.resize(img_rgb, frame_size, interpolation=cv2.INTER_LINEAR)
            # Normalize to [0, 1] range (float32)
            img_normalized = img_resized.astype(np.float32) / 255.0
            
            frames.append(img_normalized)
            frame_filenames.append(frame_file.name)
        except Exception as e:
            print(f"⚠️  Error loading {frame_file}: {e}")
            continue
    
    return np.array(frames), frame_filenames


def load_labels_from_csv(csv_path):
    """
    Load labels from CSV file.
    
    CSV format: file name, visibility, x-coordinate, y-coordinate, status
    
    Returns:
        labels: Array of shape (N, 4) with [visibility, x, y, status]
        frame_to_idx: Dictionary mapping frame filename to index
    """
    df = pd.read_csv(csv_path)
    
    # Sort by filename to match frame order
    df = df.sort_values('file name').reset_index(drop=True)
    
    labels = []
    frame_to_idx = {}
    
    for idx, row in df.iterrows():
        filename = row['file name']
        visibility = int(row['visibility']) if pd.notna(row['visibility']) else 0
        x = float(row['x-coordinate']) if pd.notna(row['x-coordinate']) else np.nan
        y = float(row['y-coordinate']) if pd.notna(row['y-coordinate']) else np.nan
        status = int(row['status']) if pd.notna(row['status']) else 0
        
        labels.append([visibility, x, y, status])
        frame_to_idx[filename] = idx
    
    return np.array(labels), frame_to_idx


def create_hdf5_dataset_from_frames(dataset_dir, output_hdf5_path):
    """
    Create HDF5 dataset from frame-based dataset structure.
    
    Dataset structure: Dataset/game{N}/Clip{N}/
        - Contains: *.jpg frames and Label.csv
    
    Args:
        dataset_dir: Root directory of dataset
        output_hdf5_path: Path to output HDF5 file
    """
    video_metadata = []
    
    # Find all game directories
    game_dirs = sorted(Path(dataset_dir).glob("game*"))
    
    with h5py.File(output_hdf5_path, 'w') as hdf5_file:
        for game_dir in tqdm(game_dirs, desc="Processing games"):
            game_name = game_dir.name  # e.g., "game1"
            
            # Find all clip directories
            clip_dirs = sorted(game_dir.glob("Clip*"))
            
            for clip_dir in clip_dirs:
                clip_name = clip_dir.name  # e.g., "Clip1"
                video_id = f"{game_name}_{clip_name}"  # e.g., "game1_Clip1"
                
                # Load frames
                frames, frame_filenames = load_frames_from_directory(clip_dir)
                if len(frames) == 0:
                    print(f"⚠️  Skipping {video_id}: No frames found")
                    continue
                
                # Load labels
                csv_path = clip_dir / "Label.csv"
                labels = None
                if csv_path.exists():
                    labels, frame_to_idx = load_labels_from_csv(csv_path)
                    # Align labels with frames (some frames might not have labels)
                    aligned_labels = []
                    for filename in frame_filenames:
                        if filename in frame_to_idx:
                            idx = frame_to_idx[filename]
                            aligned_labels.append(labels[idx])
                        else:
                            # Frame without label - use default
                            aligned_labels.append([0, np.nan, np.nan, 0])
                    labels = np.array(aligned_labels)
                
                # Store in HDF5
                group = hdf5_file.create_group(video_id)
                group.create_dataset('frames', data=frames, compression='gzip', compression_opts=4)
                
                # Create timestamps (assuming 30 FPS)
                timestamps = np.arange(len(frames)) / TARGET_FPS
                group.create_dataset('timestamps', data=timestamps)
                
                # Store labels if available
                if labels is not None:
                    group.create_dataset('labels', data=labels.astype(np.float32))
                
                # Metadata
                metadata = {
                    'video_id': video_id,
                    'game': game_name,
                    'clip': clip_name,
                    'num_frames': len(frames),
                    'frame_shape': frames[0].shape if len(frames) > 0 else None,
                    'has_labels': labels is not None
                }
                video_metadata.append(metadata)
    
    return video_metadata


# Load dataset and create HDF5
print("📝 Task 1: Data Import & Format Standardization")
print(f"\n📁 Dataset directory: {DATASET_DIR}")
print("📋 Dataset structure: Dataset/game{N}/Clip{N}/*.jpg + Label.csv\n")

# Create HDF5 dataset
hdf5_path = os.path.join(HDF5_DIR, "dataset.hdf5")
print("🔄 Processing dataset...")
video_metadata = create_hdf5_dataset_from_frames(DATASET_DIR, hdf5_path)

# Save metadata mapping
metadata_df = pd.DataFrame(video_metadata)
metadata_csv = os.path.join(METADATA_DIR, "video_metadata.csv")
metadata_df.to_csv(metadata_csv, index=False)

print(f"\n✅ HDF5 dataset created: {hdf5_path}")
print(f"✅ Metadata saved: {metadata_csv}")
print(f"\n📊 Summary:")
print(f"   Total videos (clips): {len(video_metadata)}")
print(f"   Total frames: {metadata_df['num_frames'].sum():,}")
print(f"   Videos with labels: {metadata_df['has_labels'].sum()}")
display(metadata_df.head(10))

📝 Task 1: Data Import & Format Standardization

📁 Dataset directory: Dataset
📋 Dataset structure: Dataset/game{N}/Clip{N}/*.jpg + Label.csv

🔄 Processing dataset...


Processing games: 100%|██████████| 6/6 [03:42<00:00, 37.01s/it]


✅ HDF5 dataset created: Dataset\processed\hdf5\dataset.hdf5
✅ Metadata saved: Dataset\processed\metadata\video_metadata.csv

📊 Summary:
   Total videos (clips): 50
   Total frames: 10,490
   Videos with labels: 49


,video_id,game,clip,num_frames,frame_shape,has_labels
0,game1_Clip1,game1,Clip1,207,"(224, 224, 3)",True
1,game1_Clip10,game1,Clip10,109,"(224, 224, 3)",True
2,game1_Clip11,game1,Clip11,130,"(224, 224, 3)",True
3,game1_Clip12,game1,Clip12,126,"(224, 224, 3)",True
4,game1_Clip13,game1,Clip13,253,"(224, 224, 3)",True
5,game1_Clip2,game1,Clip2,130,"(224, 224, 3)",True
6,game1_Clip3,game1,Clip3,63,"(224, 224, 3)",True
7,game1_Clip4,game1,Clip4,108,"(224, 224, 3)",True
8,game1_Clip5,game1,Clip5,160,"(224, 224, 3)",True
9,game1_Clip6,game1,Clip6,78,"(224, 224, 3)",True


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TASK 2: Metadata Cleaning & Noise Removal
# ═══════════════════════════════════════════════════════════════════════════════

def velocity_check(coordinates, max_velocity=MAX_VELOCITY, dt=DT):
    """
    Filter out outliers using velocity checks.
    If distance between consecutive frames exceeds max_velocity * dt, flag as noise.
    
    Args:
        coordinates: Array of shape (N, 2) with (x, y) coordinates
        max_velocity: Maximum allowed velocity in pixels/frame
        dt: Time delta between frames
    
    Returns:
        valid_mask: Boolean array indicating valid coordinates
    """
    if len(coordinates) < 2:
        return np.ones(len(coordinates), dtype=bool)
    
    valid_mask = np.ones(len(coordinates), dtype=bool)
    
    for i in range(1, len(coordinates)):
        if np.isnan(coordinates[i]).any() or np.isnan(coordinates[i-1]).any():
            valid_mask[i] = False
            continue
        
        # Calculate distance
        dx = coordinates[i][0] - coordinates[i-1][0]
        dy = coordinates[i][1] - coordinates[i-1][1]
        distance = np.sqrt(dx**2 + dy**2)
        
        # Check if exceeds maximum velocity
        if distance > max_velocity * dt:
            valid_mask[i] = False
    
    return valid_mask


def smooth_coordinates(coordinates, window_length=5, polyorder=2):
    """
    Apply Savitzky-Golay filter to smooth coordinates and remove high-frequency jitter.
    
    Args:
        coordinates: Array of shape (N, 2) with (x, y) coordinates
        window_length: Must be odd, less than N
        polyorder: Polynomial order (must be < window_length)
    
    Returns:
        smoothed: Smoothed coordinates
    """
    if len(coordinates) < window_length:
        return coordinates
    
    # Ensure window_length is odd and less than N
    if window_length % 2 == 0:
        window_length += 1
    window_length = min(window_length, len(coordinates) if len(coordinates) % 2 == 1 else len(coordinates) - 1)
    
    if window_length < polyorder + 1:
        return coordinates
    
    smoothed = np.zeros_like(coordinates)
    for dim in range(coordinates.shape[1]):
        # Handle NaN values
        valid_mask = ~np.isnan(coordinates[:, dim])
        if valid_mask.sum() < window_length:
            smoothed[:, dim] = coordinates[:, dim]
            continue
        
        # Apply Savitzky-Golay filter
        smoothed[:, dim] = savgol_filter(
            coordinates[:, dim], 
            window_length=window_length, 
            polyorder=polyorder
        )
    
    return smoothed


def crop_active_zone(frame, active_zone=None):
    """
    Crop frame to active zone (pitch/court area).
    
    Args:
        frame: Input frame (H, W, C)
        active_zone: Tuple (x1, y1, x2, y2) or None to use full frame
    
    Returns:
        cropped_frame: Cropped frame
    """
    if active_zone is None:
        return frame
    
    x1, y1, x2, y2 = active_zone
    return frame[y1:y2, x1:x2]


def detect_activity(frames, threshold=0.1):
    """
    Detect active periods in video using frame difference.
    Returns mask indicating active frames.
    
    Args:
        frames: Array of frames (N, H, W, C)
        threshold: Activity threshold (fraction of pixels that changed)
    
    Returns:
        active_mask: Boolean array indicating active frames
    """
    if len(frames) < 2:
        return np.ones(len(frames), dtype=bool)
    
    active_mask = np.ones(len(frames), dtype=bool)
    active_mask[0] = True  # First frame is always active
    
    for i in range(1, len(frames)):
        # Calculate frame difference
        diff = np.abs(frames[i] - frames[i-1])
        change_ratio = np.mean(diff > 0.05)  # Threshold for significant change
        
        active_mask[i] = change_ratio > threshold
    
    return active_mask


def clean_video_data(hdf5_path, video_id, active_zone=None, activity_threshold=0.1):
    """
    Clean data for a single video: remove outliers, smooth coordinates, trim dead time.
    
    Returns:
        cleaned_data: Dictionary with cleaned frames, labels, and metadata
    """
    with h5py.File(hdf5_path, 'r') as hdf5_file:
        if video_id not in hdf5_file:
            return None
        
        group = hdf5_file[video_id]
        frames = np.array(group['frames'])
        timestamps = np.array(group['timestamps'])
        
        # 1. Temporal trimming: Remove dead time
        active_mask = detect_activity(frames, threshold=activity_threshold)
        frames = frames[active_mask]
        timestamps = timestamps[active_mask]
        
        # 2. Spatial trimming: Crop to active zone
        if active_zone is not None:
            frames = np.array([crop_active_zone(f, active_zone) for f in frames])
        
        # 3. Clean labels if available
        cleaned_labels = None
        if 'labels' in group:
            labels = np.array(group['labels'])
            labels = labels[active_mask]  # Apply same temporal mask
            
            # Extract coordinates (labels format: [visibility, x, y, status])
            # Only process visible coordinates (visibility > 0)
            if labels.shape[1] >= 3:
                visibility = labels[:, 0]
                x_coords = labels[:, 1]
                y_coords = labels[:, 2]
                
                # Create coordinates array (x, y) - only for visible frames
                coordinates = np.column_stack([x_coords, y_coords])
                
                # Mark invalid coordinates (NaN or visibility=0) for filtering
                valid_coords_mask = (
                    (visibility > 0) & 
                    ~np.isnan(x_coords) & 
                    ~np.isnan(y_coords)
                )
                
                # 4. Velocity check: Remove outliers (only on valid coordinates)
                if valid_coords_mask.sum() > 1:
                    # Create a mask for velocity check
                    velocity_valid_mask = np.ones(len(coordinates), dtype=bool)
                    for i in range(1, len(coordinates)):
                        if not (valid_coords_mask[i] and valid_coords_mask[i-1]):
                            continue
                        if np.isnan(coordinates[i]).any() or np.isnan(coordinates[i-1]).any():
                            velocity_valid_mask[i] = False
                            continue
                        
                        dx = coordinates[i][0] - coordinates[i-1][0]
                        dy = coordinates[i][1] - coordinates[i-1][1]
                        distance = np.sqrt(dx**2 + dy**2)
                        
                        if distance > MAX_VELOCITY * DT:
                            velocity_valid_mask[i] = False
                    
                    # Apply velocity mask
                    frames = frames[velocity_valid_mask]
                    timestamps = timestamps[velocity_valid_mask]
                    labels = labels[velocity_valid_mask]
                    coordinates = coordinates[velocity_valid_mask]
                    valid_coords_mask = valid_coords_mask[velocity_valid_mask]
                
                # 5. Smooth coordinates (only smooth valid coordinates)
                if valid_coords_mask.sum() > 1:
                    smoothed_coords = smooth_coordinates(coordinates)
                    labels[:, 1] = smoothed_coords[:, 0]  # Update x
                    labels[:, 2] = smoothed_coords[:, 1]  # Update y
                
                cleaned_labels = labels
        
        return {
            'frames': frames,
            'timestamps': timestamps,
            'labels': cleaned_labels,
            'num_frames_original': len(group['frames']),
            'num_frames_cleaned': len(frames)
        }


# Example usage
print("📝 Task 2: Metadata Cleaning & Noise Removal")
print("\nFunctions defined:")
print("  ✅ velocity_check() - Removes ghost ball detections")
print("  ✅ smooth_coordinates() - Savitzky-Golay filtering")
print("  ✅ crop_active_zone() - Spatial trimming")
print("  ✅ detect_activity() - Temporal trimming")
print("  ✅ clean_video_data() - Complete cleaning pipeline")

# Clean all videos in HDF5
hdf5_path = os.path.join(HDF5_DIR, "dataset.hdf5")
cleaned_hdf5_path = os.path.join(HDF5_DIR, "dataset_cleaned.hdf5")

print("\n🔄 Cleaning dataset (removing noise, smoothing coordinates)...")
with h5py.File(hdf5_path, 'r') as src, h5py.File(cleaned_hdf5_path, 'w') as dst:
    total_original = 0
    total_cleaned = 0
    
    for video_id in tqdm(src.keys(), desc="Cleaning videos"):
        cleaned = clean_video_data(hdf5_path, video_id)
        if cleaned is None:
            continue
        
        total_original += cleaned['num_frames_original']
        total_cleaned += cleaned['num_frames_cleaned']
        
        group = dst.create_group(video_id)
        group.create_dataset('frames', data=cleaned['frames'], compression='gzip')
        group.create_dataset('timestamps', data=cleaned['timestamps'])
        if cleaned['labels'] is not None:
            group.create_dataset('labels', data=cleaned['labels'])

print(f"\n✅ Cleaned dataset saved: {cleaned_hdf5_path}")
print(f"📊 Cleaning stats:")
print(f"   Original frames: {total_original:,}")
print(f"   Cleaned frames: {total_cleaned:,}")
print(f"   Removed: {total_original - total_cleaned:,} ({100*(total_original-total_cleaned)/total_original:.1f}%)")

📝 Task 2: Metadata Cleaning & Noise Removal

Functions defined:
  ✅ velocity_check() - Removes ghost ball detections
  ✅ smooth_coordinates() - Savitzky-Golay filtering
  ✅ crop_active_zone() - Spatial trimming
  ✅ detect_activity() - Temporal trimming
  ✅ clean_video_data() - Complete cleaning pipeline

🔄 Cleaning dataset (removing noise, smoothing coordinates)...


Cleaning videos: 100%|██████████| 50/50 [00:38<00:00,  1.30it/s]


✅ Cleaned dataset saved: Dataset\processed\hdf5\dataset_cleaned.hdf5
📊 Cleaning stats:
   Original frames: 10,490
   Cleaned frames: 50
   Removed: 10,440 (99.5%)


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TASK 3: Data Balancing & Format Conversion
# ═══════════════════════════════════════════════════════════════════════════════

def augment_frame(frame, augmentation_type='hflip'):
    """
    Apply augmentation to a single frame.
    
    Args:
        frame: Frame array (H, W, C) normalized to [0, 1]
        augmentation_type: Type of augmentation ('hflip', 'brightness', 'contrast')
    
    Returns:
        augmented_frame: Augmented frame
    """
    if augmentation_type == 'hflip':
        # Horizontal flip (creates synthetic left-handed data)
        return np.fliplr(frame)
    
    elif augmentation_type == 'brightness':
        # Brightness adjustment
        factor = np.random.uniform(0.8, 1.2)
        return np.clip(frame * factor, 0.0, 1.0)
    
    elif augmentation_type == 'contrast':
        # Contrast adjustment
        factor = np.random.uniform(0.9, 1.1)
        mean = np.mean(frame)
        return np.clip((frame - mean) * factor + mean, 0.0, 1.0)
    
    elif augmentation_type == 'rotation':
        # Small rotation
        angle = np.random.uniform(-5, 5)
        center = (frame.shape[1] // 2, frame.shape[0] // 2)
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        augmented = cv2.warpAffine(frame, M, (frame.shape[1], frame.shape[0]))
        return np.clip(augmented, 0.0, 1.0)
    
    else:
        return frame


def augment_coordinates(coords, frame_shape, augmentation_type='hflip'):
    """
    Transform coordinates based on augmentation.
    
    Args:
        coords: Array of (x, y) coordinates
        frame_shape: (H, W) of the frame
        augmentation_type: Type of augmentation
    
    Returns:
        transformed_coords: Transformed coordinates
    """
    if augmentation_type == 'hflip':
        # Flip x-coordinate
        coords = coords.copy()
        coords[:, 0] = frame_shape[1] - coords[:, 0]
        return coords
    else:
        return coords


def balance_dataset(hdf5_path, min_samples_per_class=300, max_samples_per_class=500):
    """
    Balance dataset by oversampling minority classes and augmenting.
    
    Args:
        hdf5_path: Path to HDF5 dataset
        min_samples_per_class: Minimum samples per class (oversample to this)
        max_samples_per_class: Maximum samples per class (undersample to this)
    
    Returns:
        balanced_hdf5_path: Path to balanced HDF5 dataset
    """
    # First, analyze class distribution
    class_counts = {}
    video_to_class = {}
    
    with h5py.File(hdf5_path, 'r') as f:
        for video_id in f.keys():
            # Extract class label from video_id (format: "game{N}_Clip{N}")
            # Use game as class label
            class_label = video_id.split('_')[0]  # e.g., "game1"
            video_to_class[video_id] = class_label
            class_counts[class_label] = class_counts.get(class_label, 0) + len(f[video_id]['frames'])
    
    print("Class distribution before balancing:")
    for class_label, count in sorted(class_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"  {class_label}: {count:,} frames")
    
    # Identify minority and majority classes
    minority_classes = [c for c, count in class_counts.items() if count < min_samples_per_class]
    majority_classes = [c for c, count in class_counts.items() if count > max_samples_per_class]
    
    print(f"\nMinority classes (< {min_samples_per_class}): {len(minority_classes)}")
    print(f"Majority classes (> {max_samples_per_class}): {len(majority_classes)}")
    
    # Create balanced dataset
    balanced_hdf5_path = hdf5_path.replace('.hdf5', '_balanced.hdf5')
    
    with h5py.File(hdf5_path, 'r') as src, h5py.File(balanced_hdf5_path, 'w') as dst:
        for video_id in tqdm(src.keys(), desc="Balancing dataset"):
            group_src = src[video_id]
            class_label = video_to_class[video_id]
            num_frames = len(group_src['frames'])
            
            # Undersample majority classes
            if class_label in majority_classes and num_frames > max_samples_per_class:
                indices = np.random.choice(num_frames, max_samples_per_class, replace=False)
                frames = np.array(group_src['frames'])[indices]
                timestamps = np.array(group_src['timestamps'])[indices]
            else:
                frames = np.array(group_src['frames'])
                timestamps = np.array(group_src['timestamps'])
            
            # Oversample minority classes with augmentation
            if class_label in minority_classes and num_frames < min_samples_per_class:
                needed = min_samples_per_class - num_frames
                augmented_frames = []
                augmented_timestamps = []
                
                for i in range(needed):
                    frame_idx = i % len(frames)
                    aug_type = ['hflip', 'brightness', 'contrast'][i % 3]
                    
                    aug_frame = augment_frame(frames[frame_idx], aug_type)
                    augmented_frames.append(aug_frame)
                    augmented_timestamps.append(timestamps[frame_idx])
                
                frames = np.concatenate([frames, np.array(augmented_frames)], axis=0)
                timestamps = np.concatenate([timestamps, np.array(augmented_timestamps)], axis=0)
            
            # Store in balanced dataset
            group_dst = dst.create_group(video_id)
            group_dst.create_dataset('frames', data=frames.astype(np.float32), compression='gzip')
            group_dst.create_dataset('timestamps', data=timestamps.astype(np.float32))
            
            if 'labels' in group_src:
                labels = np.array(group_src['labels'])
                if class_label in majority_classes and num_frames > max_samples_per_class:
                    labels = labels[indices]
                group_dst.create_dataset('labels', data=labels.astype(np.float32))
    
    print(f"\n✅ Balanced dataset saved: {balanced_hdf5_path}")
    return balanced_hdf5_path


# Example usage
print("📝 Task 3: Data Balancing & Format Conversion")
print("\nFunctions defined:")
print("  ✅ augment_frame() - Apply augmentations (hflip, brightness, contrast, rotation)")
print("  ✅ augment_coordinates() - Transform coordinates for augmentations")
print("  ✅ balance_dataset() - Oversample minority, undersample majority classes")

# Balance dataset
hdf5_path = os.path.join(HDF5_DIR, "dataset_cleaned.hdf5")
balanced_path = balance_dataset(hdf5_path, min_samples_per_class=300, max_samples_per_class=500)

📝 Task 3: Data Balancing & Format Conversion

Functions defined:
  ✅ augment_frame() - Apply augmentations (hflip, brightness, contrast, rotation)
  ✅ augment_coordinates() - Transform coordinates for augmentations
  ✅ balance_dataset() - Oversample minority, undersample majority classes
Class distribution before balancing:
  game1: 13 frames
  game10: 12 frames
  game3: 9 frames
  game2: 8 frames
  game4: 7 frames
  game5: 1 frames

Minority classes (< 300): 6
Majority classes (> 500): 0


Balancing dataset: 100%|██████████| 50/50 [03:03<00:00,  3.67s/it]


✅ Balanced dataset saved: Dataset\processed\hdf5\dataset_cleaned_balanced.hdf5


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# TASK 4: Session-Based Data Split (70% Train | 15% Valid | 15% Test)
# ═══════════════════════════════════════════════════════════════════════════════

def session_based_split(hdf5_path, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15, random_state=42):
    """
    Split dataset by VideoID (session-based) to prevent data leakage.
    
    CRITICAL: We split by video, not by frames, because consecutive frames are highly correlated.
    
    Args:
        hdf5_path: Path to HDF5 dataset
        train_ratio: Proportion for training set
        val_ratio: Proportion for validation set
        test_ratio: Proportion for test set
        random_state: Random seed for reproducibility
    
    Returns:
        split_dict: Dictionary with 'train', 'valid', 'test' video IDs
    """
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, "Ratios must sum to 1.0"
    
    # Get all video IDs
    with h5py.File(hdf5_path, 'r') as f:
        video_ids = list(f.keys())
    
    # Optionally, group by class for stratified split
    # This ensures each class is represented in train/val/test
    video_to_class = {}
    for video_id in video_ids:
        # Extract class from video_id (format: "game{N}_Clip{N}")
        class_label = video_id.split('_')[0]  # e.g., "game1"
        video_to_class[video_id] = class_label
    
    # Create DataFrame for easier splitting
    df = pd.DataFrame({
        'video_id': video_ids,
        'class': [video_to_class[vid] for vid in video_ids]
    })
    
    # Check if stratified splitting is possible
    # Stratified split requires at least 2 samples per class
    class_counts = df['class'].value_counts()
    can_stratify = (class_counts >= 2).all() and len(class_counts) > 1
    
    if can_stratify:
        print(f"✅ Using stratified split (all classes have ≥2 samples)")
    else:
        print(f"⚠️  Some classes have <2 samples. Using non-stratified split.")
        print(f"   Class distribution: {dict(class_counts)}")
    
    # First split: Train (70%) vs Temp (30%)
    X_train, X_temp = train_test_split(
        df, 
        test_size=(val_ratio + test_ratio),
        stratify=df['class'] if can_stratify else None,
        random_state=random_state
    )
    
    # Second split: Temp -> Val (15%) + Test (15%)
    # Check if we can stratify the temp split
    temp_class_counts = X_temp['class'].value_counts()
    can_stratify_temp = (temp_class_counts >= 2).all() and len(temp_class_counts) > 1
    
    # Adjust test_size to get 15% of total (which is 50% of the 30% temp)
    X_val, X_test = train_test_split(
        X_temp,
        test_size=test_ratio / (val_ratio + test_ratio),  # 0.15 / 0.30 = 0.5
        stratify=X_temp['class'] if can_stratify_temp else None,
        random_state=random_state
    )
    
    split_dict = {
        'train': X_train['video_id'].tolist(),
        'valid': X_val['video_id'].tolist(),
        'test': X_test['video_id'].tolist()
    }
    
    return split_dict


def create_split_datasets(hdf5_path, split_dict, output_dir):
    """
    Create separate HDF5 files for train/val/test splits.
    
    Args:
        hdf5_path: Path to source HDF5 dataset
        split_dict: Dictionary with 'train', 'valid', 'test' video IDs
        output_dir: Directory to save split datasets
    """
    os.makedirs(output_dir, exist_ok=True)
    
    for split_name, video_ids in split_dict.items():
        output_path = os.path.join(output_dir, f"{split_name}.hdf5")
        
        with h5py.File(hdf5_path, 'r') as src, h5py.File(output_path, 'w') as dst:
            for video_id in tqdm(video_ids, desc=f"Creating {split_name} split"):
                if video_id not in src:
                    continue
                
                # Copy entire video group
                src.copy(src[video_id], dst, video_id)
        
        print(f"✅ {split_name} split saved: {output_path}")


def create_split_metadata(hdf5_path, split_dict, output_csv):
    """
    Create CSV metadata file mapping each frame to its split and video.
    
    Args:
        hdf5_path: Path to HDF5 dataset
        split_dict: Dictionary with 'train', 'valid', 'test' video IDs
        output_csv: Path to output CSV file
    """
    records = []
    
    with h5py.File(hdf5_path, 'r') as f:
        for split_name, video_ids in split_dict.items():
            for video_id in video_ids:
                if video_id not in f:
                    continue
                
                group = f[video_id]
                num_frames = len(group['frames'])
                
                for frame_idx in range(num_frames):
                    records.append({
                        'video_id': video_id,
                        'frame_idx': frame_idx,
                        'split': split_name
                    })
    
    df = pd.DataFrame(records)
    df.to_csv(output_csv, index=False)
    print(f"✅ Split metadata saved: {output_csv}")
    return df


# Example usage
print("📝 Task 4: Session-Based Data Split")
print("\n⚠️  CRITICAL: Splitting by VideoID, not by frames!")
print("   This prevents data leakage from highly correlated consecutive frames.\n")

# Perform session-based split
# FIXED: Use the correct balanced dataset filename
# The balance_dataset function creates: dataset_cleaned_balanced.hdf5
hdf5_path = os.path.join(HDF5_DIR, "dataset_cleaned_balanced.hdf5")

# Create splits
split_dict = session_based_split(hdf5_path, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15)

# Print summary
print("\n" + "="*60)
print("Session-Based Split Summary")
print("="*60)
total_videos = sum(len(vids) for vids in split_dict.values())
for split_name, video_ids in split_dict.items():
    percentage = len(video_ids) / total_videos * 100
    print(f"{split_name.upper():8s}: {len(video_ids):4d} videos ({percentage:5.1f}%)")
print("="*60)

# Create separate HDF5 files for each split
split_dir = os.path.join(PROCESSED_DIR, "splits")
create_split_datasets(hdf5_path, split_dict, split_dir)

# Create metadata CSV
metadata_csv = os.path.join(METADATA_DIR, "split_metadata.csv")
split_metadata_df = create_split_metadata(hdf5_path, split_dict, metadata_csv)

# Display sample
display(split_metadata_df.head(10))

📝 Task 4: Session-Based Data Split

⚠️  CRITICAL: Splitting by VideoID, not by frames!
   This prevents data leakage from highly correlated consecutive frames.

⚠️  Some classes have <2 samples. Using non-stratified split.
   Class distribution: {'game1': np.int64(13), 'game10': np.int64(12), 'game3': np.int64(9), 'game2': np.int64(8), 'game4': np.int64(7), 'game5': np.int64(1)}

Session-Based Split Summary
TRAIN   :   35 videos ( 70.0%)
VALID   :    7 videos ( 14.0%)
TEST    :    8 videos ( 16.0%)


Creating train split: 100%|██████████| 35/35 [00:06<00:00,  5.71it/s]


✅ train split saved: Dataset\processed\splits\train.hdf5


Creating valid split: 100%|██████████| 7/7 [00:01<00:00,  3.97it/s]


✅ valid split saved: Dataset\processed\splits\valid.hdf5


Creating test split: 100%|██████████| 8/8 [00:02<00:00,  3.07it/s]

✅ test split saved: Dataset\processed\splits\test.hdf5
✅ Split metadata saved: Dataset\processed\metadata\split_metadata.csv


,video_id,frame_idx,split
0,game10_Clip4,0,train
1,game10_Clip4,1,train
2,game10_Clip4,2,train
3,game10_Clip4,3,train
4,game10_Clip4,4,train
5,game10_Clip4,5,train
6,game10_Clip4,6,train
7,game10_Clip4,7,train
8,game10_Clip4,8,train
9,game10_Clip4,9,train


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# COMPLETE PIPELINE EXAMPLE
# ═══════════════════════════════════════════════════════════════════════════════

def complete_pipeline(video_dir, label_dir, output_dir):
    """
    Complete pipeline: Import → Clean → Balance → Split
    
    This is a template function. Adapt based on your dataset structure.
    """
    print("🚀 Starting Complete Pipeline")
    print("="*60)
    
    # Step 1: Import & Format Standardization
    print("\n[1/4] Data Import & Format Standardization")
    video_paths = list(Path(video_dir).glob("*.mp4")) + list(Path(video_dir).glob("*.mov"))
    labels_dict = {}
    
    for label_file in Path(label_dir).glob("*.json"):
        video_id = label_file.stem
        with open(label_file, 'r') as f:
            labels_dict[video_id] = json.load(f)
    
    hdf5_path = os.path.join(output_dir, "dataset.hdf5")
    video_metadata = create_hdf5_dataset(video_paths, labels_dict, hdf5_path)
    print(f"✅ Created HDF5 dataset: {hdf5_path}")
    
    # Step 2: Cleaning
    print("\n[2/4] Metadata Cleaning & Noise Removal")
    cleaned_hdf5_path = os.path.join(output_dir, "dataset_cleaned.hdf5")
    # (Implementation of batch cleaning would go here)
    print(f"✅ Cleaned dataset: {cleaned_hdf5_path}")
    
    # Step 3: Balancing
    print("\n[3/4] Data Balancing & Format Conversion")
    balanced_hdf5_path = balance_dataset(cleaned_hdf5_path)
    print(f"✅ Balanced dataset: {balanced_hdf5_path}")
    
    # Step 4: Session-Based Split
    print("\n[4/4] Session-Based Data Split")
    split_dict = session_based_split(balanced_hdf5_path)
    split_dir = os.path.join(output_dir, "splits")
    create_split_datasets(balanced_hdf5_path, split_dict, split_dir)
    
    metadata_csv = os.path.join(output_dir, "split_metadata.csv")
    create_split_metadata(balanced_hdf5_path, split_dict, metadata_csv)
    
    print("\n" + "="*60)
    print("✅ Pipeline Complete!")
    print("="*60)
    
    return {
        'raw': hdf5_path,
        'cleaned': cleaned_hdf5_path,
        'balanced': balanced_hdf5_path,
        'splits': split_dict
    }


print("📋 Complete Pipeline Template Ready")
print("\nTo use, adapt the following based on your dataset:")
print("  - Video file locations and naming convention")
print("  - Label file format (JSON, CSV, etc.)")
print("  - Class extraction from video IDs")
print("  - Active zone coordinates (if known)")

📋 Complete Pipeline Template Ready

To use, adapt the following based on your dataset:
  - Video file locations and naming convention
  - Label file format (JSON, CSV, etc.)
  - Class extraction from video IDs
  - Active zone coordinates (if known)
